### Training Reasoning Models with Reinforcement Learning

In [1]:
from pathlib import Path
import sys
import torch

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from evaluating_reasoning_models.model_and_tokenizer import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    use_compile=False
)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [2]:
from improving_reasoning_with_inference_time_scaling.improving_reasoning_with_inference_time_scaling import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache,
)
from evaluating_reasoning_models.evaluating_reasoning_models import render_prompt

device = "cuda" if torch.cuda.is_available() else "cpu"
raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)




prompt = render_prompt(raw_prompt)
print(prompt)

torch.manual_seed(0)
response_1 = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=1048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.2,
    top_p=0.9,
)


You are a helpful math assistant.

Solve the problem step by step.
The last line of your response should contain only the final answer inside \boxed{}.

Question:
Half the value of $3x-9$ is $x+37$. What is the value of $x$?

Answer:

\boxed{x=12}



#### Loading a Math training subset

In [3]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        "MATH/math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            print("Using backup URL.")
            r = requests.get(backup_url, timeout=30)
            r.raise_for_status()

        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data


In [ ]:

math_train = load_math_train()

print("Dataset size:", len(math_train))

Dataset size: 12000


In [5]:
from pprint import pprint

pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

#### Sampling rollouts

In [6]:
from base_model.qwen import KVCache
from improving_reasoning_with_inference_time_scaling.improving_reasoning_with_inference_time_scaling import top_p_filter


@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for _ in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        generated.append(token_id)

        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break
        logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [7]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

\boxed{x=37}

Step-by-Step:

We are given that half the value of $3x - 9$ is $x + 37$. So, what is the value of $x$?

First, we set up the equation: $\frac{1}{2}(3x - 9) = x + 37$.

We then solve the equation to find $x$.

Step 1:
$$
\frac{1}{2}(3x - 9) = x + 37
$$

Step 2:
Multiply both sides by 2 to eliminate the fraction: $3x - 9 = 2x + 74$

Step 3:
Subtract $2x$ from both sides: $x - 9 = 74$

Step 4:
Add 9 to both sides: $x = 83$

Step 5:
Subtract 37 from both sides: $x = 46$

Wait, but the answer is different. I need to check my steps again.

Step 1:
$$
\frac{1}{2}(3x - 9) = x + 37
$$

Step 2:
Multiply both sides by 2: $3x - 9 = 2x + 74$

Step 3:
Subtract $2x$ from both sides: $x - 9 = 74$

Step 4:
Add 9 to both sides: $x = 83$

Wait, but earlier I had x=46. Now I have x=83. Which one is correct?

I must have made a mistake in my first solution. Let me check again.

Original problem: Half the value of $3x - 9$ is $x + 37$. So, half of (3x -9) equals x +37.

So, half of (3x -9) is 

In [8]:
torch.manual_seed(5)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

\boxed{x=11}

So, what is the value of $x$?

Answer:
\boxed{x=11}

So, what is the value of $x$?

Answer:
\boxed{x=11}

So, what is the value of $x$?

Answer:
\boxed{x=11}

So, what is the value of $x$?

Answer:
\boxed{x=11}

Okay, let's see. The problem says that half the value of $3x - 9$ is equal to $x + 37$. I need to find the value of $x$.

First, I should translate the problem into an equation. The problem states that half of $3x - 9$ is $x + 37$. So, half of $3x - 9$ means dividing that expression by 2. So, if I take half of $3x - 9$, that would be $\frac{1}{2}(3x - 9)$. According to the problem, this is equal to $x + 37$.

So, setting up the equation: $\frac{1}{2}(3x - 9) = x + 37$.

Now, I need to solve this equation for $x$. Let me start by eliminating the fraction. To do that, I can multiply both sides of the equation by 2. That way, I can get rid of the denominator. 

Multiplying both sides by 2: $2 \times \frac{1}{2}(3x - 9) = 2(x + 37)$.

Simplifying the left side, the 2 

In [9]:
### example rollouts
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

#### Calculating rewards

In [10]:
from evaluating_reasoning_models.evaluating_reasoning_models import (extract_final_candidate,
grade_answer)

def reward_rlvr(answer_text,  ground_truth):
    extracted = extract_final_candidate(answer_text, fallback=None)

    if not extracted:
        return 0.0
    
    correct = grade_answer(extracted, ground_truth)
    return float(correct)


In [11]:

rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

Answer: '\\boxed{83}'
Reward: 1.0

Answer: 'The correct answer is \\boxed{83}'
Reward: 1.0

Answer: 'The final answer is 83'
Reward: 0.0

Answer: 'We get \\boxed{38}'
Reward: 0.0



#### Learning signals from rollouts via advantages

In [12]:
rewards = torch.tensor(rollout_rewards, device=device)
print(rewards)

tensor([1., 1., 0., 0.], device='cuda:0')


In [13]:
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([ 0.8659,  0.8659, -0.8659, -0.8659], device='cuda:0')


#### Scoring rollouts with sequence log-probabilites

In [ ]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    # Positions whose next-token probabilities we want
    # These correspond to predicting token_ids[t + 1] from position t
    start = prompt_len - 1
    end = token_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Sum log-probabilities over the answer tokens
    return torch.sum(next_token_logps)

print(sequence_logprob_draft(model, token_ids, prompt_len))

** using torch.gather function

In [ ]:
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

In [ ]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )

    print(f"Answer:  {text}")
    print(f"Logprob: {logprob.item():.4f}\n")

    rollout_logps.append(logprob)

#### from advatages to policy-updates via a grpo loss